# Review Sentiment Analysis
**Ecommerce AI Series - Nub Labs Playbooks**

Classify customer review sentiment, identify product quality issues, and surface the product aspects driving negative feedback.

Dataset: [TechHeaven Reference Platform](https://github.com/Nub-Labs/techheaven-reference-platform) -
5,000 reviews across 291 products (Jan 2025 - Jun 2026). No API key required.

## 1. Business problem

TechHeaven has 5,000 customer reviews. A retailer reading them manually would take weeks. Reading the average rating per product misses the pattern inside the text: a 3-star review that says "the product works but the app keeps disconnecting" and a 3-star review that says "decent but overpriced for what you get" are very different signals. The first is a firmware or software quality issue. The second is a pricing perception problem. The average rating cannot distinguish them.

Sentiment analysis reads the text, scores it, and answers three questions:

1. **Which products have hidden problems?** Products with moderate ratings but consistently negative comment text, or products with high average ratings masking a vocal minority of serious complaints.
2. **What are customers actually complaining about?** Aspect extraction - finding which product features (battery life, connectivity, build quality, app experience) appear most often in negative reviews - turns qualitative feedback into an actionable engineering or procurement signal.
3. **Is satisfaction improving or declining?** A monthly sentiment trend per category shows whether product quality perception is improving after a firmware update or declining after a supplier change.

This playbook uses VADER (Valence Aware Dictionary and sEntiment Reasoner), a rule-based sentiment model that requires no training data and no API key. VADER was designed for short consumer text - product reviews, social media, forum posts. It produces a compound polarity score from -1 (maximally negative) to +1 (maximally positive) in milliseconds per review.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install vaderSentiment pandas numpy matplotlib seaborn -q

## 2. The dataset

Three files from the TechHeaven public repository:

- `reviews/reviews.json` - 5,000 reviews: review_id, product_id, customer_id, rating (1-5), title, comment, review_date
- `catalog/products.json` - 291 products: product_id, name, category, brand, price

Joining on `product_id` gives each review a product category. All 5,000 reviews are in `approved` status.

**Rating distribution:** skewed positive (2,250 five-star reviews vs 150 one-star). This is typical of ecommerce review data - the silent majority does not review, and satisfied customers review more than dissatisfied ones. Sentiment analysis on the text identifies negative signal within the high-rating majority that average ratings miss.

In [ ]:
import json
import urllib.request
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

BASE = 'https://raw.githubusercontent.com/Nub-Labs/techheaven-reference-platform/main/data'

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'
sns.set_style('whitegrid')

In [ ]:
def fetch_json(url: str) -> list:
    with urllib.request.urlopen(url) as r:
        return json.load(r)

reviews_raw  = fetch_json(f'{BASE}/reviews/reviews.json')
products_raw = fetch_json(f'{BASE}/catalog/products.json')

reviews  = pd.DataFrame(reviews_raw)
products = pd.DataFrame(products_raw)

print(f'Reviews:  {len(reviews):,}')
print(f'Products: {len(products):,}')
print(f'Fields:   {list(reviews.columns)}')

In [ ]:
# Join reviews to product catalog to get category per review
df = reviews.merge(
    products[['product_id', 'name', 'category', 'brand']],
    on='product_id',
    how='left'
)
df['review_date'] = pd.to_datetime(df['review_date'])
df['text'] = df['title'].fillna('') + '. ' + df['comment'].fillna('')

print(f'Date range: {df["review_date"].min().date()} to {df["review_date"].max().date()}')
print(f'Categories: {df["category"].nunique()}')
print(f'\nRating distribution:')
print(df['rating'].value_counts().sort_index().to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
counts = df['rating'].value_counts().sort_index()
colors = ['#ef4444', '#f97316', '#eab308', '#22c55e', '#10b981']
bars = ax.bar(counts.index, counts.values, color=colors, width=0.6, edgecolor='white')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20, f'{val:,}',
            ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Review rating distribution - TechHeaven', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Star rating')
ax.set_ylabel('Number of reviews')
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(['1 star\n(negative)', '2 stars', '3 stars', '4 stars', '5 stars\n(positive)'])
plt.tight_layout()
plt.savefig('rating_distribution.png', bbox_inches='tight')
plt.show()

## 3. VADER sentiment scoring

VADER (Valence Aware Dictionary and sEntiment Reasoner) is a rule-based sentiment model optimised for short consumer text. It does not require training - it ships with a pre-built lexicon of ~7,500 words annotated with valence scores by human raters.

VADER produces four scores for any piece of text:
- `pos` - proportion of text with positive valence
- `neg` - proportion of text with negative valence
- `neu` - proportion of text with neutral valence
- `compound` - normalised aggregate score from -1 (most negative) to +1 (most positive)

**Classification thresholds (standard VADER):**
- compound >= 0.05 → positive
- compound <= -0.05 → negative
- between -0.05 and 0.05 → neutral

A compound score of +0.6 or higher is reliably positive. A compound score of -0.4 or lower is reliably negative. Most 3-star reviews fall in the 0.0 to 0.4 range - mixed or mildly positive.

In [ ]:
# Score all 5,000 reviews
sia = SentimentIntensityAnalyzer()

scores = df['text'].apply(lambda t: sia.polarity_scores(str(t)))
df['vader_compound'] = scores.apply(lambda s: s['compound'])
df['vader_pos']      = scores.apply(lambda s: s['pos'])
df['vader_neg']      = scores.apply(lambda s: s['neg'])
df['vader_neu']      = scores.apply(lambda s: s['neu'])

# Classify
df['sentiment'] = df['vader_compound'].apply(
    lambda c: 'positive' if c >= 0.05 else ('negative' if c <= -0.05 else 'neutral')
)

print(f'Scored: {len(df):,} reviews')
print(f'\nSentiment distribution:')
print(df['sentiment'].value_counts().to_string())
print(f'\nAverage compound by star rating:')
print(df.groupby('rating')['vader_compound'].mean().round(3).to_string())

In [ ]:
# VADER compound vs star rating - shows correlation between text sentiment and numeric rating
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: scatter (sample to avoid overplotting)
sample = df.sample(min(1500, len(df)), random_state=42)
colors_map = {'positive': '#10b981', 'neutral': '#94a3b8', 'negative': '#ef4444'}
for sentiment, group in sample.groupby('sentiment'):
    axes[0].scatter(group['rating'], group['vader_compound'],
                    alpha=0.25, s=12, c=colors_map[sentiment], label=sentiment)
axes[0].set_xlabel('Star rating')
axes[0].set_ylabel('VADER compound score')
axes[0].set_title('Star rating vs VADER compound score', fontweight='bold')
axes[0].set_xticks([1, 2, 3, 4, 5])
axes[0].legend(markerscale=2, fontsize=9)

# Right: box plot per rating
df.boxplot(column='vader_compound', by='rating', ax=axes[1],
           boxprops=dict(color='#6366f1'),
           medianprops=dict(color='#ef4444', linewidth=2),
           whiskerprops=dict(color='#6366f1'),
           capprops=dict(color='#6366f1'))
axes[1].set_title('VADER compound distribution by star rating', fontweight='bold')
axes[1].set_xlabel('Star rating')
axes[1].set_ylabel('VADER compound score')
plt.suptitle('')

corr, pval = stats.pearsonr(df['rating'], df['vader_compound'])
fig.suptitle(f'Pearson r = {corr:.3f} (p < 0.001)', fontsize=10, y=1.01)
plt.tight_layout()
plt.savefig('vader_vs_rating.png', bbox_inches='tight')
plt.show()
print(f'Pearson correlation: r = {corr:.3f}, p-value = {pval:.2e}')

In [ ]:
# Alignment between VADER label and star rating label
# Star rating positive = 4 or 5 stars; negative = 1 or 2 stars; neutral = 3 stars
df['rating_label'] = df['rating'].apply(
    lambda r: 'positive' if r >= 4 else ('negative' if r <= 2 else 'neutral')
)

aligned   = (df['sentiment'] == df['rating_label']).sum()
misaligned = (df['sentiment'] != df['rating_label']).sum()

print(f'VADER vs star rating alignment:')
print(f'  Aligned:    {aligned:,} ({aligned/len(df)*100:.1f}%)')
print(f'  Misaligned: {misaligned:,} ({misaligned/len(df)*100:.1f}%)')
print()

# Misalignment breakdown - where do VADER and star ratings disagree?
print('Misalignment breakdown:')
cross = pd.crosstab(df['rating_label'], df['sentiment'],
                    rownames=['Star rating category'], colnames=['VADER label'])
print(cross)
print()

# Reviews where customer gave 1 star but VADER reads as neutral or positive.
# These are cases where the complaint language is mixed or indirect.
one_star_positive = df[
    (df['rating'] == 1) & (df['vader_compound'] > 0)
].sort_values('vader_compound', ascending=False).head(3)

if len(one_star_positive):
    print(f'1-star reviews where VADER reads positive (indirect complaints):')
    for _, row in one_star_positive.iterrows():
        print(f"  1★ (compound {row['vader_compound']:.2f}) - {row['name'][:30]} ({row['category']})")
        print(f"  Comment: {row['comment'][:130]}")
        print()
else:
    print('All 1-star reviews score negative on VADER - strong alignment at the extremes.')
    one_star_avg = df[df['rating'] == 1]['vader_compound'].mean()
    print(f'Average VADER compound for 1-star reviews: {one_star_avg:.3f}')

## 4. Product and category sentiment dashboard

Aggregate VADER compound scores per product and per category to identify where quality problems are concentrated. The compound score average per category is the primary signal: categories with consistently low compound scores have systemic quality or value issues, not isolated product defects.

In [ ]:
# Average VADER compound per category
cat_sentiment = (
    df.groupby('category')
    .agg(
        avg_compound=('vader_compound', 'mean'),
        avg_rating=('rating', 'mean'),
        review_count=('review_id', 'count'),
        pct_negative=('sentiment', lambda x: (x == 'negative').mean() * 100),
    )
    .sort_values('avg_compound')
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: VADER compound per category
colors = ['#ef4444' if v < 0.3 else '#f97316' if v < 0.5 else '#10b981'
          for v in cat_sentiment['avg_compound']]
bars = axes[0].barh(cat_sentiment['category'], cat_sentiment['avg_compound'],
                     color=colors, edgecolor='white')
axes[0].axvline(cat_sentiment['avg_compound'].mean(), color='#94a3b8',
                linestyle='--', linewidth=1, label=f'Overall avg ({cat_sentiment["avg_compound"].mean():.2f})')
axes[0].set_title('Average VADER compound score by category', fontweight='bold')
axes[0].set_xlabel('VADER compound (higher = more positive)')
axes[0].legend(fontsize=9)

# Right: % negative reviews per category
cat_sorted_neg = cat_sentiment.sort_values('pct_negative', ascending=False)
colors2 = ['#ef4444' if v > 10 else '#f97316' if v > 5 else '#10b981'
           for v in cat_sorted_neg['pct_negative']]
axes[1].barh(cat_sorted_neg['category'], cat_sorted_neg['pct_negative'],
              color=colors2, edgecolor='white')
axes[1].set_title('% negative sentiment reviews by category', fontweight='bold')
axes[1].set_xlabel('% reviews with negative VADER score')

plt.tight_layout()
plt.savefig('category_sentiment.png', bbox_inches='tight')
plt.show()

print('Category sentiment summary:')
print(cat_sentiment[['category', 'avg_compound', 'pct_negative', 'review_count']].to_string(index=False))

In [ ]:
# Bottom 10 products by average VADER compound (with >= 5 reviews)
product_sentiment = (
    df.groupby(['product_id', 'name', 'category'])
    .agg(
        avg_compound=('vader_compound', 'mean'),
        avg_rating=('rating', 'mean'),
        review_count=('review_id', 'count'),
        pct_negative=('sentiment', lambda x: (x == 'negative').mean() * 100),
    )
    .reset_index()
)
qualified = product_sentiment[product_sentiment['review_count'] >= 5]
bottom10 = qualified.sort_values('avg_compound').head(10)

fig, ax = plt.subplots(figsize=(13, 5))
colors = ['#ef4444' if v < 0.2 else '#f97316' for v in bottom10['avg_compound']]
y_labels = [f"{row['name'][:30]}\n({row['category']}, n={int(row['review_count'])})"
            for _, row in bottom10.iterrows()]
ax.barh(y_labels, bottom10['avg_compound'], color=colors, edgecolor='white')
ax.set_title('Bottom 10 products by VADER sentiment (min 5 reviews)', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Average VADER compound score')
plt.tight_layout()
plt.savefig('product_sentiment_bottom10.png', bbox_inches='tight')
plt.show()

print('\nBottom 10 products:')
print(bottom10[['name', 'category', 'avg_compound', 'avg_rating', 'pct_negative', 'review_count']].to_string(index=False))

## 5. Aspect-based sentiment: what are customers complaining about?

Average sentiment scores tell you WHERE the problems are. Aspect extraction tells you WHAT the problems are. Consumer electronics reviews mention specific product attributes - battery life, connectivity, build quality, display quality, the companion app. Counting how often each aspect appears in negative reviews surfaces actionable engineering signals.

The approach is lexicon-based: a dictionary of keywords per aspect, then a keyword match in negative review text. More sophisticated approaches (fine-tuned ABSA models) exist, but keyword matching produces actionable results at zero training cost and full transparency.

In [ ]:
# Aspect keyword lists for consumer electronics reviews
ASPECT_KEYWORDS = {
    'battery':       ['battery', 'charge', 'charging', 'power', 'drain', 'runtime'],
    'connectivity':  ['wifi', 'bluetooth', 'connection', 'disconnect', 'pairing', 'drops', 'signal', 'wireless'],
    'build quality': ['build', 'quality', 'flimsy', 'cheap', 'plastic', 'robust', 'durable', 'breaks', 'fragile'],
    'performance':   ['slow', 'lag', 'freeze', 'crash', 'performance', 'fast', 'speed', 'sluggish'],
    'app / software':['app', 'software', 'firmware', 'update', 'interface', 'ui', 'bug', 'glitch'],
    'value':         ['price', 'overpriced', 'expensive', 'worth', 'value', 'cost', 'cheap'],
    'display':       ['screen', 'display', 'brightness', 'resolution', 'glare', 'panel'],
    'audio':         ['sound', 'audio', 'noise', 'volume', 'bass', 'microphone', 'speakers'],
    'setup':         ['setup', 'install', 'configuration', 'instructions', 'manual', 'difficult'],
}

# Count negative reviews mentioning each aspect
neg_reviews = df[df['sentiment'] == 'negative'].copy()
neg_reviews['text_lower'] = neg_reviews['text'].str.lower()

aspect_counts = {}
for aspect, keywords in ASPECT_KEYWORDS.items():
    pattern = '|'.join(keywords)
    count = neg_reviews['text_lower'].str.contains(pattern, regex=True).sum()
    aspect_counts[aspect] = count

aspect_df = pd.DataFrame(list(aspect_counts.items()), columns=['aspect', 'negative_mentions'])
aspect_df = aspect_df.sort_values('negative_mentions', ascending=False)

print(f'Negative reviews: {len(neg_reviews):,}')
print()
print('Aspect mentions in negative reviews:')
print(aspect_df.to_string(index=False))

In [ ]:
# Aspect mentions chart + per-category breakdown for top aspects
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overall aspect counts
colors = ['#ef4444' if v > 50 else '#f97316' if v > 20 else '#94a3b8'
          for v in aspect_df['negative_mentions']]
axes[0].barh(aspect_df['aspect'], aspect_df['negative_mentions'], color=colors, edgecolor='white')
axes[0].set_title('Aspects mentioned in negative reviews', fontweight='bold')
axes[0].set_xlabel('Number of negative reviews mentioning aspect')

# Right: top 3 aspects per category
top_aspects = aspect_df.head(3)['aspect'].tolist()
cat_aspect = {}
for aspect in top_aspects:
    keywords = ASPECT_KEYWORDS[aspect]
    pattern = '|'.join(keywords)
    by_cat = (
        neg_reviews[neg_reviews['text_lower'].str.contains(pattern, regex=True)]
        .groupby('category').size()
        .sort_values(ascending=False)
    )
    cat_aspect[aspect] = by_cat

combined = pd.DataFrame(cat_aspect).fillna(0)
combined.plot(kind='barh', ax=axes[1], width=0.7, edgecolor='white')
axes[1].set_title(f'Top 3 aspects by category (negative reviews)', fontweight='bold')
axes[1].set_xlabel('Negative review count')
axes[1].legend(title='Aspect', fontsize=8)

plt.tight_layout()
plt.savefig('aspect_sentiment.png', bbox_inches='tight')
plt.show()

In [ ]:
# Monthly sentiment trend per category - top 5 categories by volume
top5_cats = df.groupby('category')['review_id'].count().sort_values(ascending=False).head(5).index.tolist()

df['month'] = df['review_date'].dt.to_period('M').dt.to_timestamp()

monthly_trend = (
    df[df['category'].isin(top5_cats)]
    .groupby(['month', 'category'])['vader_compound']
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 4.5))
palette = sns.color_palette('husl', 5)
for cat, color in zip(top5_cats, palette):
    cat_data = monthly_trend[monthly_trend['category'] == cat].sort_values('month')
    ax.plot(cat_data['month'], cat_data['vader_compound'],
            marker='o', markersize=3, linewidth=2, label=cat, color=color)

ax.axhline(0.05, color='#94a3b8', linestyle='--', linewidth=1, label='Neutral threshold (0.05)')
ax.set_title('Monthly average VADER compound score - top 5 categories', fontsize=12, fontweight='bold', pad=10)
ax.set_ylabel('VADER compound score')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45, ha='right')
ax.legend(fontsize=9, loc='lower right')
plt.tight_layout()
plt.savefig('sentiment_trend.png', bbox_inches='tight')
plt.show()

In [ ]:
# Most negative reviews - qualitative examples of what to action
worst = df.nsmallest(5, 'vader_compound')[['name', 'category', 'rating', 'vader_compound', 'title', 'comment']]

print('Most negative reviews (lowest VADER compound):')
print()
for _, row in worst.iterrows():
    print(f"{row['rating']}★ | compound {row['vader_compound']:.3f} | {row['name'][:35]} ({row['category']})")
    print(f"  Title: {row['title']}")
    print(f"  Comment: {row['comment'][:150]}")
    print()

## 6. Business interpretation

### What star ratings miss

The correlation between VADER compound score and star rating is strong (typically r > 0.7), but the residual is where the value lies. Reviews with high star ratings but low compound scores contain hidden complaints: a 4-star review that praises the product but mentions the app keeps disconnecting, a 5-star review that says "great product but the battery drains fast on standby." These are engineering signals masked by a positive overall impression.

The misalignment analysis (Section 3, negative text with high rating) surfaces these cases. In a real ecommerce operation, these reviews would be routed to the product team for the specific category - not to customer service, who handle the unhappy customers, but to the product or sourcing team, who can address the root cause.

### Aspect extraction as an engineering input

The aspect chart shows which product attributes appear most often in negative reviews. If "connectivity" is the top aspect in negative reviews for Smart Home devices, that is a Wi-Fi firmware issue that affects a specific SKU or firmware version. "Battery" appearing in negative Laptop reviews after a specific date could indicate a batch quality issue. "App / software" appearing across Audio and Wearables categories suggests a platform-level app problem rather than individual product defects.

### Sentiment trend as an early warning system

The monthly trend chart shows whether satisfaction is improving or declining. A sudden drop in VADER compound score for a specific category in a specific month - before it shows up in the average star rating - gives the operations team 2-4 weeks of lead time to investigate. The lag exists because low ratings accumulate slowly; negative text is written immediately.

### What a production pipeline looks like

1. **Weekly batch job** - score new reviews with VADER, update product and category dashboards
2. **Alert integration** - flag products whose average compound score dropped >0.2 points month-over-month
3. **Aspect report** - weekly email to product team: top 3 complaint aspects per category
4. **Root cause routing** - connectivity complaints to engineering, value complaints to pricing, app complaints to mobile team
5. **LLM escalation** - for the worst 1% of reviews, use a language model to summarise the exact complaint and proposed resolution

In [ ]:
# Final summary
total = len(df)
pct_pos = (df['sentiment'] == 'positive').mean() * 100
pct_neg = (df['sentiment'] == 'negative').mean() * 100
pct_neu = (df['sentiment'] == 'neutral').mean() * 100

from scipy import stats as scipy_stats
corr, _ = scipy_stats.pearsonr(df['rating'], df['vader_compound'])

worst_cat = cat_sentiment.iloc[0]
top_aspect = aspect_df.iloc[0]

print('Key findings')
print(f'  Reviews scored:          {total:,}')
print(f'  Positive sentiment:      {pct_pos:.1f}%')
print(f'  Neutral sentiment:       {pct_neu:.1f}%')
print(f'  Negative sentiment:      {pct_neg:.1f}%')
print(f'  VADER-rating correlation: r = {corr:.3f}')
print(f'  Lowest-sentiment category: {worst_cat["category"]} (compound {worst_cat["avg_compound"]:.3f})')
print(f'  Top complaint aspect:     {top_aspect["aspect"]} ({int(top_aspect["negative_mentions"])} negative reviews)')
print()
print('Notebook:    github.com/Nub-Labs/playbooks')
print('Walkthrough: nublabs.com/playbooks/ecommerce/review-sentiment')